# Order Flow Microstructure: 4D Regime Detection via KDE

## What Is Order Flow Microstructure?

**Order flow** = the stream of buy/sell orders hitting the market. Professional traders analyze it to understand:
- Who is trading (institutions vs retail)?
- Are they buying or selling aggressively?
- Is volume confirming the move or diverging?
- Is liquidity thin (prone to fast moves) or thick (stable)?

**Level 2 / DOM (Depth of Market)** shows resting limit orders at each price level — the "book" of pending orders. Imbalances in the book predict short-term direction.

**What we're doing here:** Since yfinance doesn't provide tick-level L2 data, we construct **proxy features** from 1-min bars that capture the same information:

| Feature | What it proxies | Range |
|---------|----------------|-------|
| **Net Pressure** = (close-open)/range | Aggressor side (buyer vs seller dominated) | -1 to +1 |
| **Volume Intensity** = vol/avg_vol | Trade intensity (institutional participation?) | 0 to 5+ |
| **Relative Range** = range/ATR | Liquidity/volatility regime | 0 to 3+ |
| **VWAP Skew** = (VWAP-mid)/range | Where volume concentrated within bar | -0.5 to +0.5 |

Each bar is a **4-dimensional point**. The KDE finds clusters in this 4D space = distinct "bar types" or microstructure regimes.

## Ticker: NVDA | Date: 2026-08-20 | Bars: 1950 | Dimensions: 4

Bandwidth comparison:
- Silverman: h = 0.3688 → **78 regimes**
- GSJ (data-driven): h = 0.2628 → **215 regimes**


## Figure 1: Regimes in Feature Space (PCA Projection)

4D → 2D via PCA for visualization. Each point = one 1-min bar. Colors = detected regime.

![Regimes](of_fig1_regimes.png)

GSJ finds more distinct clusters because its tighter bandwidth doesn't merge nearby regime types. Silverman over-smooths and lumps different bar types together.


## Figure 2: Feature Distributions by Regime

How each regime differs across the 4 features. Each color = one regime type.

![Features](of_fig2_features.png)

Clear separation: "Aggressive Buy" bars have high net pressure + high volume, while "Quiet" bars have low range + low volume. These are genuinely different microstructure states.


## Figure 3: Price Colored by Regime (Last Trading Day)

The price chart with each bar colored by its detected microstructure regime.

![Timeline](of_fig3_timeline.png)

You can SEE regime changes: consolidation periods (one color) give way to momentum (another color). The regime transitions often correspond to key intraday events (opening drive, lunch lull, power hour).


## Figure 4: Regime Transition Matrix

"If the current bar is regime X, what's the probability the next bar is regime Y?"

![Transitions](of_fig4_transitions.png)

**Key patterns:**
- High self-transition probability on the diagonal = regimes persist for multiple bars
- Off-diagonal hotspots = common transitions (e.g., "Quiet" → "Aggressive Buy" = breakout)
- This is directly tradeable: if you detect "Absorption" followed by "Momentum Up" starting, that's a long entry signal

## Why This Is d=4 KDE (Not Just Clustering)

You could use k-means or DBSCAN for clustering. The KDE approach adds:
1. **Density estimation** — you get the probability of each regime, not just labels
2. **Bandwidth matters** — too smooth = you miss the absorption/momentum distinction; too tight = noise
3. **Anomaly detection** — bars with low density across ALL clusters = unusual microstructure events (news, halt, flash crash)
4. **No pre-specified k** — the number of regimes emerges from the data via the bandwidth choice

The bandwidth IS the resolution: it determines whether "accumulation" and "distribution" are one regime or two.


## Connection to Real Order Flow Tools

| What pros use | What our proxy captures | Limitation |
|---------------|------------------------|------------|
| Footprint charts (bid×ask volume) | Volume intensity + pressure | Can't separate bid/ask |
| Delta (buy vol - sell vol) | Net pressure (close-open direction) | Approximate only |
| DOM imbalance | VWAP skew (proxy) | No actual L2 depth |
| CVD (cumulative volume delta) | Cumulative net pressure | Not tick-precise |
| Time & Sales tape | Volume intensity + range | No individual trade sizes |

**With real tick data** (Polygon, Bookmap, exchange feeds), you'd replace our proxies with:
- Actual trade aggressor side (buy vs sell market orders)
- Real bid-ask spread at time of trade
- Order book imbalance ratio
- Inter-trade duration

The KDE method is identical — just with better input features. The bandwidth selection problem is the same in d=4 or d=8.
